# PyGeoFetch — 01: Getting Started

**PyGeoFetch** is a universal satellite data pipeline with access to 22+ providers.  
This notebook covers installation, verification, your first search, and your first download — all using **free providers** that require no credentials.

---
### What you'll learn
- Install PyGeoFetch and verify with `doctor`
- Understand the 10 free (no-login) providers
- Run your first satellite search
- Inspect search results (STAC 1.0)
- Download your first Sentinel-2 scene
- Explore the Python API

## 1. Installation

In [1]:
# Install PyGeoFetch
# Uncomment and run once
# !pip install PyGeoFetch

# With geo extras (rasterio, geopandas — for post-processing and GeoParquet output)
# !pip install "PyGeoFetch[geo]"

# Verify version
import pygeofetch
print(f"PyGeoFetch version: {pygeofetch.__version__}")

PyGeoFetch version: 1.0.0


In [ ]:
# Run the built-in doctor to check your installation
# This checks Python version, packages, keyring, and live connectivity
!PyGeoFetch doctor

## 2. Free Providers — No Credentials Needed

In [3]:
from pygeofetch.providers import list_provider_info, get_free_providers
import pandas as pd

# Get all provider metadata
all_providers = list_provider_info()
free_ids = set(get_free_providers())

# Display free providers in a table
free = [
    {
        "ID": p["id"],
        "Name": p["display_name"],
        "SAR": "✓" if p.get("supports_sar") else "—",
        "<1m": "✓" if p.get("supports_sub_meter") else "—",
        "STAC": "✓" if p.get("stac") else "—",
        "Description": p.get("description", "")[:60] + "..."
    }
    for p in all_providers if p["id"] in free_ids
]

df = pd.DataFrame(free)
print(f"Free providers ({len(df)}):")
df

Free providers (10):


,ID,Name,SAR,<1m,STAC,Description
0,aws_earth,AWS Earth Search,—,—,✓,"Free access to Sentinel-2, Landsat, NAIP, and ..."
1,digitalglobe,DigitalGlobe Open Data,—,✓,—,DigitalGlobe disaster response open data. Free...
2,element84,Element 84 Earth Search,✓,—,✓,Production STAC API by Element 84 with Sentine...
3,esa_scihub,ESA Copernicus Hub Mirror,✓,—,—,Mirror access to Copernicus Sentinel data. No ...
4,geoserver_generic,GeoServer Generic,—,—,—,Generic OGC WMS/WFS/WCS GeoServer endpoint. Co...
5,inpe_cbers,INPE CBERS,—,—,—,CBERS-4 and CBERS-4A data from INPE. Free publ...
6,isro_bhuvan,ISRO Bhuvan,—,—,—,"Indian satellite data: ResourceSat, Cartosat, ..."
7,jaxa_earth,JAXA ALOS World,✓,—,—,"ALOS World 3D DSM, ALOS PALSAR global mosaics...."
8,noaa_big_data,NOAA Big Data,—,—,—,"GOES-16/17/18, NEXRAD, MRMS, and other NOAA da..."
9,planetary_computer,Microsoft Planetary Computer,✓,—,✓,Free STAC catalog from Microsoft with Sentinel...


## 3. Your First Search — Python API

In [4]:
from pygeofetch import PyGeoFetch
from pygeofetch.models.search_query import SearchQuery, BoundingBox

# Initialize the client
client = PyGeoFetch(log_level="WARNING")  # Suppress INFO logs in notebook

# Define search area: New York City bounding box
# Format: min_lon, min_lat, max_lon, max_lat  (longitude first!)
nyc_bbox = BoundingBox.from_string("-74.1,40.6,-73.7,40.9")
print(f"Search bbox: {nyc_bbox}")
print(f"  Width:  {nyc_bbox.max_lon - nyc_bbox.min_lon:.2f}°")
print(f"  Height: {nyc_bbox.max_lat - nyc_bbox.min_lat:.2f}°")

Search bbox: min_lon=-74.1 min_lat=40.6 max_lon=-73.7 max_lat=40.9
  Width:  0.40°
  Height: 0.30°


In [5]:
# Build the search query
query = SearchQuery(
    bbox=nyc_bbox,
    start_date="2024-01-01",
    end_date="2024-06-01",
    cloud_cover_max=15,      # Max 15% cloud cover
    max_results=50,
    sort_by="cloud_cover",   # Best (least cloudy) scenes first
    sort_ascending=True,
)

# Search AWS Earth Search (free, no login, Sentinel-2 + Landsat)
print("Searching AWS Earth Search...")
results = client.search(query, providers=["aws_earth"])
print(f"Found {len(results)} scenes")

Searching AWS Earth Search...
Found 50 scenes


## 4. Explore Results

In [6]:
# Look at the top results
print("Top 5 scenes (lowest cloud cover):")
print(f"{'ID':<35} {'Satellite':<15} {'Date':<12} {'Cloud%':>7} {'Score':>7}")
print("-" * 80)
for r in results[:5]:
    date = str(r.properties.get("datetime", ""))[:10] if r.properties else "—"
    cloud = f"{r.cloud_cover:.0f}" if r.cloud_cover is not None else "—"
    score = f"{r.score:.3f}" if r.score else "—"
    print(f"{r.id:<35} {str(r.satellite):<15} {date:<12} {cloud:>7} {score:>7}")

Top 5 scenes (lowest cloud cover):
ID                                  Satellite       Date          Cloud%   Score
--------------------------------------------------------------------------------
S2B_18TXL_20240524_0_L2A            sentinel-2b     2024-05-24         0   0.700
S2B_18TXK_20240531_0_L2A            sentinel-2b     2024-05-31         0   0.699
S2B_18TWL_20240524_0_L2A            sentinel-2b     2024-05-24         0   0.699
S2B_18TXK_20240501_0_L2A            sentinel-2b     2024-05-01         1   0.697
S2B_18TWK_20240524_0_L2A            sentinel-2b     2024-05-24         1   0.697


In [7]:
# Inspect a single result in detail
scene = results[0]
print(f"Scene ID:    {scene.id}")
print(f"Provider:    {scene.provider}")
print(f"Satellite:   {scene.satellite}")
print(f"Sensor:      {scene.sensor}")
print(f"Collection:  {scene.collection}")
print(f"Cloud cover: {scene.cloud_cover}%")
print(f"Score:       {scene.score:.4f}")
print(f"Bbox:        {scene.bbox}")
print(f"Assets ({len(scene.assets)}):  {list(scene.assets.keys())}")
print(f"Data assets: {list(scene.data_assets.keys())}")

Scene ID:    S2B_18TXL_20240524_0_L2A
Provider:    aws_earth
Satellite:   sentinel-2b
Sensor:      msi
Collection:  sentinel-2-l2a
Cloud cover: 0.088447%
Score:       0.6996
Bbox:        (-73.81879130623679, 40.55949675802678, -73.45708803321031, 41.54558463811687)
Assets (35):  ['aot', 'blue', 'coastal', 'granule_metadata', 'green', 'nir', 'nir08', 'nir09', 'red', 'rededge1', 'rededge2', 'rededge3', 'scl', 'swir16', 'swir22', 'thumbnail', 'tileinfo_metadata', 'visual', 'wvp', 'aot-jp2', 'blue-jp2', 'coastal-jp2', 'green-jp2', 'nir-jp2', 'nir08-jp2', 'nir09-jp2', 'red-jp2', 'rededge1-jp2', 'rededge2-jp2', 'rededge3-jp2', 'scl-jp2', 'swir16-jp2', 'swir22-jp2', 'visual-jp2', 'wvp-jp2']
Data assets: ['aot', 'blue', 'coastal', 'green', 'nir', 'nir08', 'nir09', 'red', 'rededge1', 'rededge2', 'rededge3', 'scl', 'swir16', 'swir22', 'wvp', 'aot-jp2', 'blue-jp2', 'coastal-jp2', 'green-jp2', 'nir-jp2', 'nir08-jp2', 'nir09-jp2', 'red-jp2', 'rededge1-jp2', 'rededge2-jp2', 'rededge3-jp2', 'scl-jp2'

In [8]:
# Convert to STAC GeoJSON and inspect structure
stac_feature = scene.to_stac_item()
import json
print("STAC Feature (abridged):")
print(json.dumps({
    "type": stac_feature["type"],
    "id": stac_feature["id"],
    "properties_keys": list(stac_feature.get("properties", {}).keys())[:10],
    "asset_keys": list(stac_feature.get("assets", {}).keys()),
    "bbox": stac_feature.get("bbox"),
}, indent=2))

STAC Feature (abridged):
{
  "type": "Feature",
  "id": "S2B_18TXL_20240524_0_L2A",
  "properties_keys": [
    "created",
    "platform",
    "constellation",
    "instruments",
    "eo:cloud_cover",
    "proj:epsg",
    "mgrs:utm_zone",
    "mgrs:latitude_band",
    "mgrs:grid_square",
    "grid:code"
  ],
  "asset_keys": [
    "aot",
    "blue",
    "coastal",
    "granule_metadata",
    "green",
    "nir",
    "nir08",
    "nir09",
    "red",
    "rededge1",
    "rededge2",
    "rededge3",
    "scl",
    "swir16",
    "swir22",
    "thumbnail",
    "tileinfo_metadata",
    "visual",
    "wvp",
    "aot-jp2",
    "blue-jp2",
    "coastal-jp2",
    "green-jp2",
    "nir-jp2",
    "nir08-jp2",
    "nir09-jp2",
    "red-jp2",
    "rededge1-jp2",
    "rededge2-jp2",
    "rededge3-jp2",
    "scl-jp2",
    "swir16-jp2",
    "swir22-jp2",
    "visual-jp2",
    "wvp-jp2"
  ],
  "bbox": [
    -73.81879130623679,
    40.55949675802678,
    -73.45708803321031,
    41.54558463811687
  ]
}


In [9]:
# Build a DataFrame from results for easy analysis
import pandas as pd

records = []
for r in results:
    records.append({
        "id": r.id,
        "provider": r.provider,
        "satellite": r.satellite,
        "date": str(r.properties.get("datetime", ""))[:10] if r.properties else "",
        "cloud_cover": r.cloud_cover,
        "score": round(r.score, 4) if r.score else None,
        "n_assets": len(r.assets),
    })

df = pd.DataFrame(records)
print(f"Results summary:")
print(df.describe(include='all').loc[["count", "unique", "mean", "min", "max"]].T)
print("\nSatellite breakdown:")
print(df["satellite"].value_counts())

Results summary:
            count unique       mean       min     max
id             50     50        NaN       NaN     NaN
provider       50      1        NaN       NaN     NaN
satellite      50      2        NaN       NaN     NaN
date           50     13        NaN       NaN     NaN
cloud_cover  50.0    NaN  61.430978  0.088447   100.0
score        50.0    NaN   0.454274       0.3  0.6996
n_assets     50.0    NaN       35.0      35.0    35.0

Satellite breakdown:
satellite
sentinel-2b    26
sentinel-2a    24
Name: count, dtype: int64


In [10]:
# Save results to GeoJSON — use this file for downloads
from pathlib import Path
Path("output").mkdir(exist_ok=True)

client.searcher.save_results(results, Path("output/nyc_results.geojson"))
print(f"Saved {len(results)} results → output/nyc_results.geojson")

# Also export as STAC FeatureCollection
fc = client.searcher.to_geojson(results)
with open("output/nyc_results_stac.json", "w") as f:
    json.dump(fc, f, indent=2, default=str)
print(f"Saved STAC FeatureCollection → output/nyc_results_stac.json")

Saved 50 results → output/nyc_results.geojson
Saved STAC FeatureCollection → output/nyc_results_stac.json


## 5. Your First Download

In [11]:
from pygeofetch.models.download_task import DownloadOptions

# Download only the RGB bands to keep it small (~150MB vs 600MB for full scene)
options = DownloadOptions(
    parallel=1,
    retry_attempts=3,
    resume=True,
    verify_checksum=False,
    on_failure="skip",
    bands=["B02", "B03", "B04"],  # Blue, Green, Red
)

# Download the single best scene (lowest cloud cover)
best_scene = results[0]
print(f"Downloading: {best_scene.id}")
print(f"  Cloud cover: {best_scene.cloud_cover}%")
print(f"  Bands: B02 (Blue), B03 (Green), B04 (Red)")
print()

dl_results = client.download(
    [best_scene],
    destination=Path("output/downloads/"),
    options=options
)

for dr in dl_results:
    if dr.success:
        size_mb = dr.bytes_downloaded / (1024 * 1024) if dr.bytes_downloaded else 0
        print(f"✓ {dr.data_id}")
        print(f"  Size: {size_mb:.1f} MB")
        print(f"  Duration: {dr.duration_seconds:.1f}s")
        print(f"  Speed: {dr.speed_mbps:.2f} Mbps")
        if dr.output_paths:
            for p in dr.output_paths:
                print(f"  File: {p}")
    else:
        print(f"✗ Failed: {dr.error}")

Downloading: S2B_18TXL_20240524_0_L2A
  Cloud cover: 0.088447%
  Bands: B02 (Blue), B03 (Green), B04 (Red)



KeyboardInterrupt: 

In [ ]:
# Check what was downloaded
import os
dl_dir = Path("output/downloads")
if dl_dir.exists():
    all_files = list(dl_dir.rglob("*.tif"))
    print(f"Downloaded {len(all_files)} file(s):")
    for f in all_files:
        size_mb = f.stat().st_size / (1024 * 1024)
        print(f"  {f.name}  ({size_mb:.1f} MB)")

## 6. System Status

In [ ]:
# Check system status
status = client.status()
print(f"PyGeoFetch v{status['version']}")
print(f"Authenticated providers: {status['providers_authenticated'] or 'none'}")
print(f"Free providers: {len(status['providers_free'])} available")
print(f"Search cache entries: {status['cache_entries']}")

# Clear cache if needed
# client.clear_cache()

In [ ]:
# CLI equivalents of everything we did above
print("CLI equivalents:")
print()
print("# Install")
print("pip install PyGeoFetch")
print()
print("# Diagnose")
print("PyGeoFetch doctor")
print()
print("# Search")
print('PyGeoFetch search run \\')
print('  --bbox "-74.1,40.6,-73.7,40.9" \\')
print('  --start-date 2024-01-01 --end-date 2024-06-01 \\')
print('  --cloud-cover 0-15 \\')
print('  --providers aws_earth \\')
print('  --format table \\')
print('  --output output/nyc_results.geojson')
print()
print("# Download")
print('PyGeoFetch download run \\')
print('  --from-search output/nyc_results.geojson \\')
print('  --output output/downloads/ \\')
print('  --max-items 1 \\')
print('  --bands "B02,B03,B04"')

---
## Summary
You've successfully:
- ✅ Installed and verified PyGeoFetch
- ✅ Searched AWS Earth Search (free) for Sentinel-2 scenes over NYC
- ✅ Inspected STAC 1.0 compliant results
- ✅ Saved results to GeoJSON
- ✅ Downloaded RGB bands (~150 MB) of the clearest scene

**Next:** `02_authentication_and_providers.ipynb` — add credentials for USGS, Copernicus, Planet, and more.